# Equity, commodity and diversified ETF pair pool: local V2

Build two-asset candidates without an A/B/C requirement. Keep up to 300 equity pairs, up to 50 commodity ETF/ETP pairs and up to 50 other ETF pairs. ETF exposure buckets have separate caps, so highly correlated index clones cannot consume the entire pool. This stage opens no positions.

`etf_universe.json` contains a curated US-listed USD catalog covering bullion, energy, agricultural and industrial-metal futures, broad commodities, equity indices/sectors and bonds. Leveraged/inverse products, ETNs and closed-end funds are outside this catalog. An instrument listed in the catalog is not automatically accepted.

The legacy A/B/C notebooks and their two CSVs remain unchanged. This workflow updates only `data/pair_pool.csv`. No price cache or intermediate CSV is saved. Work remains on local branch `research/pair-pool-v2`; GitHub `v1.0.0` preserves the old version.

## Screening and timing

- Yahoo market caps, ETF assets under management (AUM), classifications and catalog membership are current snapshots, not reconstructed 2025 constituents. They can introduce survivorship/selection bias.
- All correlation, OLS, EG, residual ADF, spread and historical liquidity calculations use only **2022-2025 Close/Volume**. No 2026 price data enter this stage. All prices remain Close, not adjusted Close; dividend cash flows remain excluded. Fund distributions and futures rolls can affect Close-based spreads.
- Equities require current market cap >= USD 10 billion. ETFs require current AUM >= USD 100 million; equity market cap is not substituted for AUM. Both legs require a formation-end price >= USD 5 and a final-60-observation average Close * Volume >= USD 20 million. One whole share must fit the USD 100,000 reference budget.
- Known equity share classes are reduced to the class with higher formation liquidity. ETFs tracking the same index or physical metal remain candidates but are marked `tracking_pair=True` where catalogued. This flag neither establishes tax eligibility nor demonstrates a profitable spread after costs.
- Equity pairs must share an industry or have a configured related industry. ETF pairs must share a catalog exposure group or a configured related group. No stock/ETF cross-pairs, or commodity/bond cross-pairs, are formed. Mining and energy-company ETFs are equity exposures, distinct from bullion/futures ETPs.
- Correlation > 0.90; EG uses 95% critical values and residual ADF uses fixed -2.86. Both use lag 1 with autolag=None. ETF levels must additionally support I(1): ADF does not reject at the level and does reject after differencing. This prevents stationary cash-like price series from being treated as price-level cointegration.
- For a USD 100,000 illustrative long slot, both trading directions must imply orders <= 0.5% of historical average daily dollar volume at formation-end prices. This is a screening proxy, not a slippage/borrow-availability model. Actual execution conditions need separate checks.
- Physical trusts, futures pools and investment-company ETFs can have different tax treatment. This pool does not calculate tax or establish wash-sale/straddle treatment.
- Each family has its own target; ETF exposure-group pairs are capped at five accepted candidates. Unfilled targets are left unfilled, and missing/illiquid funds are reported rather than forced into pairs.

In [1]:
from pathlib import Path
import logging
import pandas as pd
from IPython.display import display
from portfolio import CointegrationConfig
from pair_pool import PairPoolConfig, PairPoolBuilder

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

In [2]:
config = PairPoolConfig(
    target_pairs=300,                      # Equity pair target
    target_commodity_pairs=50,             # Additional commodity ETF/ETP pairs, if eligible
    target_other_etf_pairs=50,              # Additional equity/bond ETF pairs, if eligible
    max_etf_pairs_per_group=5,
    include_equities=True,
    include_etfs=True,
    require_etf_i1=True,
    max_assets=1000,                       # Maximum equity universe; ETF catalog is separate
    min_market_cap=10_000_000_000,
    min_etf_assets=100_000_000,             # ETF AUM, not market cap
    min_average_dollar_volume=20_000_000,
    liquidity_window=60,
    min_price=5.0,
    reference_long_amount=100_000,
    max_order_fraction_adv=0.005,
    min_correlation=0.90,
    confidence_level=0.95,
    batch_size=20,
    cointegration=CointegrationConfig(
        train_start="2022-01-01",
        train_end="2025-12-31",
        min_observations=750,
        max_lag=1,
        check_i1=False,                    # Existing equity policy; ETFs use require_etf_i1 above
    ),
    relationship_file=Path("industry_relationships.json"),
    etf_catalog_file=Path("etf_universe.json"),
    output_file=Path("data/pair_pool.csv"),
)
print("Equity market-cap minimum:", f"${config.min_market_cap:,.0f}")
print("ETF AUM minimum:", f"${config.min_etf_assets:,.0f}")
print("Historical daily dollar-volume minimum:", f"${config.min_average_dollar_volume:,.0f}")

Equity market-cap minimum: $10,000,000,000
ETF AUM minimum: $100,000,000
Historical daily dollar-volume minimum: $20,000,000


In [3]:
builder = PairPoolBuilder(config)
result = builder.build()
pair_pool = result["pairs"]

print("Size-screened instruments:", len(result["universe"]))
print("Formation-eligible instruments:", len(result["eligible_assets"]))
print("Correlated industry/exposure candidates:", result["candidate_count"])
print("Pairs tested:", result["tested_count"])
print("Accepted pairs:", len(pair_pool))
print("Saved:", result["output_file"])
family_summary = pd.DataFrame({"target": result["targets"], "accepted": result["family_counts"]}).fillna(0).astype(int)
display(family_summary)
preview = ["pair_id", "pair_family", "exposure_group_y", "exposure_group_x", "alpha", "beta", "correlation", "eg_pvalue", "tracking_pair"]
for family in ["equity", "commodity_etf", "other_etf"]:
    print(family)
    display(pair_pool.loc[pair_pool["pair_family"].eq(family), preview].head(15))
if result["events"]:
    print("Source event count:", len(result["events"]))
    print("First source events:", result["events"][:10])

INFO: Industry screens 10/145; 67 equity records


INFO: Industry screens 20/145; 177 equity records


INFO: Industry screens 30/145; 226 equity records


INFO: Industry screens 40/145; 290 equity records


INFO: Industry screens 50/145; 344 equity records


...

commodity_etf


,pair_id,pair_family,exposure_group_y,exposure_group_x,alpha,beta,correlation,eg_pvalue,tracking_pair
0,IAU__OUNZ,commodity_etf,gold_physical,gold_physical,-0.009318,1.956890,0.999999,0.000000e+00,True
2,SGOL__AAAU,commodity_etf,gold_physical,gold_physical,-0.007121,0.965824,0.999999,0.000000e+00,True
3,AAAU__BAR,commodity_etf,gold_physical,gold_physical,0.009135,1.001303,0.999999,1.344318e-29,True
4,SGOL__BAR,commodity_etf,gold_physical,gold_physical,0.001683,0.967084,0.999999,0.000000e+00,True
5,IAU__AAAU,commodity_etf,gold_physical,gold_physical,0.123930,1.905078,0.999999,1.684167e-22,True
6,SLV__SIVR,commodity_etf,silver_physical,silver_physical,0.179411,0.948925,0.999992,3.071882e-12,True


other_etf


,pair_id,pair_family,exposure_group_y,exposure_group_x,alpha,beta,correlation,eg_pvalue,tracking_pair
1,QQQ__QQQM,other_etf,nasdaq100,nasdaq100,0.473926,2.427123,0.999999,6.985769e-23,True
7,SPY__SPYM,other_etf,us_sp500,us_sp500,0.634819,8.490488,0.999992,1.808430e-26,True
8,VOO__SPYM,other_etf,us_sp500,us_sp500,0.370366,7.809748,0.999989,5.568244e-22,True
9,VGT__FTEC,other_etf,technology_equity,technology_equity,0.629178,0.416834,0.999986,4.193534e-07,False
10,IVV__SPY,other_etf,us_sp500,us_sp500,-0.036623,1.004839,0.999985,3.864791e-21,True
11,VOO__SPY,other_etf,us_sp500,us_sp500,-0.207852,0.919812,0.999985,9.249418e-20,True
12,VOO__IVV,other_etf,us_sp500,us_sp500,-0.166594,0.915367,0.999983,1.852224e-14,True
13,VTI__ITOT,other_etf,us_total_market,us_total_market,0.481739,2.250079,0.999981,1.188712e-14,False
14,ITOT__SCHB,other_etf,us_total_market,us_total_market,0.485833,5.646566,0.999980,3.915226e-15,False
15,VTI__SCHB,other_etf,us_total_market,us_total_market,1.570766,12.705435,0.999977,9.503519e-15,False


Source event count: 15
First source events: ['missing_history SPCX: No closing prices', 'missing_history SKHY: No closing prices', 'missing_history HONA: No closing prices', 'missing_history VYLR: No closing prices', 'missing_history SUNB: No closing prices', 'missing_history BSP: No closing prices', 'missing_history CBRS: No closing prices', 'missing_history ARXS: No closing prices', 'missing_history PS: No closing prices', 'missing_history FDXF: No closing prices']


## Formation spread and future signals

For each accepted row, `spread = Close_y - alpha - beta * Close_x` and `z = (spread - spread_mean) / spread_std`. The saved estimates are formation-only. This stage does not open trades or choose entry/exit thresholds.

`reference_participation_y/x` are the worse of the two reference directions. `pair_family`, instrument type, exposure group, implementation, AUM, size timestamps and issuer-reference URLs are kept in the same final CSV. ETF coverage below includes categories with zero accepted pairs so a broad catalog is not confused with guaranteed tradable opportunities.

A later strategy should limit active pairs and aggregate ticker/industry/exposure risk, allow fewer than ten positions, and account for execution costs. Commodity ETP tax rules and fund distributions need their own modelling; stock-only tax assumptions are not automatically transferable.

In [4]:
assert pair_pool["pair_id"].is_unique
assert pair_pool["y_ticker"].ne(pair_pool["x_ticker"]).all()
assert len({frozenset(row) for row in pair_pool[["y_ticker", "x_ticker"]].itertuples(index=False, name=None)}) == len(pair_pool)
assert pair_pool["price_basis"].eq("Close").all()
assert pair_pool["used_lag"].eq(1).all()
assert pair_pool["formation_end"].le("2025-12-31").all()
assert pair_pool["correlation"].gt(config.min_correlation).all()
assert pair_pool["eg_statistic"].lt(pair_pool["eg_critical_value"]).all()
assert pair_pool["residual_adf_statistic"].lt(-2.86).all()
assert pair_pool["beta"].gt(0).all()
for side in ["y", "x"]:
    equity = pair_pool[f"instrument_type_{side}"].eq("EQUITY")
    etf = pair_pool[f"instrument_type_{side}"].eq("ETF")
    assert pair_pool.loc[equity, f"market_cap_{side}"].ge(config.min_market_cap).all()
    assert pair_pool.loc[etf, f"fund_assets_{side}"].ge(config.min_etf_assets).all()
    if config.require_etf_i1:
        assert pair_pool.loc[etf, f"level_adf_statistic_{side}"].ge(-2.86).all()
        assert pair_pool.loc[etf, f"difference_adf_statistic_{side}"].lt(-2.86).all()
    assert pair_pool[f"average_dollar_volume_{side}"].ge(config.min_average_dollar_volume).all()
    assert pair_pool[f"reference_participation_{side}"].le(config.max_order_fraction_adv).all()
assert pair_pool["instrument_type_y"].eq(pair_pool["instrument_type_x"]).all()
print("Verified: unique two-asset candidates, formation-only Close, lag 1, EG/ADF and instrument-specific size/liquidity filters.")

Verified: unique two-asset candidates, formation-only Close, lag 1, EG/ADF and instrument-specific size/liquidity filters.


In [5]:
display(pd.Series(result["asset_rejections"], name="asset_count").sort_values(ascending=False).to_frame())
display(pd.Series(result["pair_rejections"], name="pair_count").sort_values(ascending=False).to_frame())
display(pair_pool.groupby(["pair_family", "scope"]).size().rename("pairs").to_frame())
print("Distinct accepted-pool assets:", len(set(pair_pool["y_ticker"]) | set(pair_pool["x_ticker"])))
print("Largest reference order / historical daily dollar volume:",
      f"{pair_pool[['reference_participation_y', 'reference_participation_x']].max().max():.2%}")
print("ETF coverage: zero means that the catalog category did not produce an accepted pair under these thresholds.")
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(result["etf_coverage"])

,asset_count
dollar_volume_below_minimum,75
ETF_price_not_supported_as_I1,49
insufficient_training_history,36
missing_close_or_volume,15
price_below_minimum,6
ETF_missing_or_small_AUM,6
whole_share_budget_too_small,1
invalid_formation_volume,1


,pair_count
cointegration_not_significant;residual_adf_not_stationary,521
cointegration_not_significant,269
reference_order_exceeds_adv_limit,3


pairs
pair_family   scope                      
commodity_etf same_etf_exposure         6
equity        related_industry        173
              same_industry           127
other_etf     related_etf_exposure     15
              same_etf_exposure        35

Distinct accepted-pool assets: 279
Largest reference order / historical daily dollar volume: 0.49%
ETF coverage: zero means that the catalog category did not produce an accepted pair under these thresholds.


,pair_family,exposure_group,catalog_funds,metadata_funds,size_eligible_funds,formation_eligible_funds,accepted_pairs_involving_group
0,commodity_etf,agriculture_basket_futures,3,3,2,0,0
1,commodity_etf,battery_metals_futures,1,1,0,0,0
2,commodity_etf,broad_commodity_futures,12,12,12,2,0
3,commodity_etf,copper_futures,1,1,1,0,0
4,commodity_etf,corn_futures,1,1,1,0,0
5,commodity_etf,crude_oil_futures,4,4,3,0,0
6,commodity_etf,energy_basket_futures,1,1,1,0,0
7,commodity_etf,gasoline_futures,1,1,1,0,0
8,commodity_etf,gold_physical,8,8,8,8,5
9,commodity_etf,industrial_metals_futures,1,1,1,0,0
